In [1]:
import torch
import numpy as np
from utils import midpoint_to_box, upscale_box_tensor

## Midpoints

The scence comes with a mid-point representation. Each cell corresponds to the y, x coordinates (in Polar Stereographic) of the midpoint of the grid cell.

In [2]:
hr_scene_vel_tensor = torch.load("./torch_data/scene_vel_tensor.pt")

# vel has 2 pixel channels (index 0 and 1)
# Third channel (index 2) are y coords
hr_scene_vel_tensor[2, :, :]

# Forth channel (index 3) are x coords
hr_scene_vel_tensor[3, :, :]

tensor([[  350.,   800.,  1250.,  ..., 21500., 21950., 22400.],
        [  350.,   800.,  1250.,  ..., 21500., 21950., 22400.],
        [  350.,   800.,  1250.,  ..., 21500., 21950., 22400.],
        ...,
        [  350.,   800.,  1250.,  ..., 21500., 21950., 22400.],
        [  350.,   800.,  1250.,  ..., 21500., 21950., 22400.],
        [  350.,   800.,  1250.,  ..., 21500., 21950., 22400.]],
       dtype=torch.float64)

## Box-channels

**Box-channels** are the representation for grid box cells we will use to calculate covariances for non-subset lower-resolution grids expressed through high-resolution base covariance grids.

Box-channels have dimensitionality [4, H, W] where for now H = W. 
The four channels have the fixed order
- y_min
- y_max
- x_min
- x_max

Next we convert from a cell **midpoint representation to a box representation**.

In [3]:
# Use function from utils
hr_scene_vel_box_tensor = midpoint_to_box(hr_scene_vel_tensor[2:4, :, :]) # index 2 and 3

### Upscale pixel values AND adjust box values

In [4]:
scence_vel_pixel_box_tensor = torch.cat((hr_scene_vel_tensor[:2, :, :], hr_scene_vel_box_tensor),
                                        dim = 0)

# upscale by 2 and check
lr_scene_vel_pixel_box_tensor = upscale_box_tensor(scence_vel_pixel_box_tensor, upscaling_factor = 2)

# Check if it works as expected.
# Upscaled to a 900m grid
for i in range(0, int(lr_scene_vel_pixel_box_tensor.shape[0])): 
    print(f"Channel: {i}")
    print(scence_vel_pixel_box_tensor[i, 0:2, 0:2])
    print(lr_scene_vel_pixel_box_tensor[i, 0, 0])

Channel: 0
tensor([[2.6226, 2.0436],
        [1.8224, 1.8224]], dtype=torch.float64)
tensor(2.0778, dtype=torch.float64)
Channel: 1
tensor([[-1.5353,  0.5726],
        [-1.5353,  0.5726]], dtype=torch.float64)
tensor(-0.4813, dtype=torch.float64)
Channel: 2
tensor([[-557225., -557225.],
        [-557675., -557675.]], dtype=torch.float64)
tensor(-557675., dtype=torch.float64)
Channel: 3
tensor([[-556775., -556775.],
        [-557225., -557225.]], dtype=torch.float64)
tensor(-556775., dtype=torch.float64)
Channel: 4
tensor([[125., 575.],
        [125., 575.]], dtype=torch.float64)
tensor(125., dtype=torch.float64)
Channel: 5
tensor([[ 575., 1025.],
        [ 575., 1025.]], dtype=torch.float64)
tensor(1025., dtype=torch.float64)


In [5]:
lr_scene_vel_pixel_box_tensor[2:, :, :].shape

torch.Size([4, 25, 25])

## Task

**HR Input**: HR base_covar with box-channels [1 + 4 + 4, hr_dim, r_dim]  
- HR is the surface elevation. 
- Base covariance but at least space the domain of LR
- 500m resolution

**LR Input**: LR box-channels [4, lr_dim, lr_dim]
- It was upscaled to a 900m reoslution (from a natural 450m resolution)

Outputs:
- k_ah_al: high-low [hr_dim^2, lr_dim^2]
- k_al_al: low-low [lr_dim^2, lr_dim^2]

Example:
- hr_dim^2 = 46^2 = 2116
- lr_dim^2 = 25^2 = 625

In [6]:
scene_sur_midpoints46_tensor = torch.load("./torch_data/scene_sur_midpoints46_tensor.pt")
base_covar46 = torch.load("./torch_data/base_covar46.pt")
# box grid of surface but also of bed. Not using bed here.
scene_sur_box46_tensor = midpoint_to_box(scene_sur_midpoints46_tensor)

In [7]:
hr_dims_flat = 46**2

# create copys for all columns ([4, 2116, 1]) -> ([4, 2116, 2116]) and all rows
# flatten first for pariwise covariance representation
# all column vectors of row_box channels are the same because these correspond to rows. [BC, R, C]: box-channels, rows, columns
row_box_channels = scene_sur_box46_tensor.reshape(4, -1).unsqueeze(-1).repeat(1, 1, hr_dims_flat)
# .unsqueeze(-2) creates explicit dimension we wanna copy across (middle dimension)
column_box_channels = scene_sur_box46_tensor.reshape(4, -1).unsqueeze(-2).repeat(1, hr_dims_flat, 1)

# Asserting behaviour
# row_box_channels[:, :, 2] == row_box_channels[:, :, 22]
# column_box_channels[:, joker1, :] == column_box_channels[:, joker2, :]

base_covar_box = torch.cat((base_covar46.unsqueeze(0), row_box_channels, column_box_channels), dim = 0)
# Channels explained:
# 0: pixel covar
# 1: y_min for row: stay constant for one row, across all columns: [1, any, :] are all the same. Could pick [1, 0, joker] to represent column 0.
# 2: y_max for row
# 3: x_min for row
# 4: x_max for row
# 5: y_min for column: stay contant for one column, across all rows: [5, :, any] is a vector with all the same value. Could pick [5, joker, 0] to represent column 0.
# 6: y_max for column
# 7: x_min for column
# 8: x_max for column

In [51]:
base_covar_box.shape

torch.Size([9, 2116, 2116])

In [61]:
base_covar_box[5, :, 10]

tensor([-557250., -557250., -557250.,  ..., -557250., -557250., -557250.])

In [64]:
lr_box_tensor = lr_scene_vel_pixel_box_tensor[2:, :, :]
lr = lr_box_tensor.reshape(4, -1)

In [18]:
def compute_k_ah_al_rectilinear(hr_matrix, lr_2D):

    ### Check that hr_matrix (base_covariance matrix) covers sufficient area
    # y_min: rows and columns contain the same so choose row. Could find min based on on position but use torch.min instead
    if ((torch.min(lr_2D[0, :, :]) < torch.min(hr_matrix[1, :, :])) or # min of y_min
        (torch.max(lr_2D[1, :, :]) > torch.max(hr_matrix[2, :, :])) or # max of y_max
        (torch.min(lr_2D[2, :, :]) < torch.min(hr_matrix[3, :, :])) or # min of x_min
        (torch.max(lr_2D[3, :, :]) > torch.max(hr_matrix[4, :, :]))): # max of x_max
        print("We have an issue. The lr area is not covered by the hr area")
    
    # Number of cells spanned by a_l might vary. Thus tensor strucuture not ideal
    # dict for each output covariance value with area (for weighting) and with values

    # Flatten lr shape
    lr = lr_2D.reshape(4, -1)

    # Extract dimensionalities of each for the loop
    hr_dims_flat = np.array(hr_matrix.shape)[-1]
    lr_dims_flat = np.array(lr).shape[-1]

    # Create two dictionaries with empty lists that you can append to
    covariance_dict = {(hr_index, lr_index): [] for hr_index in range(0, hr_dims_flat) for lr_index in range(0, lr_dims_flat)}
    weights_dict = {(hr_index, lr_index): [] for hr_index in range(0, hr_dims_flat) for lr_index in range(0, lr_dims_flat)}

    joker = 0

    for i in range(0, lr_dims_flat):
        # Check overlap between lr and "columns" of hr (last 4 channels) of hr_matrix
        # both axis (y and x) need to overlap for there to be an area.
        for j in range(0, hr_dims_flat):
            # First: check y overlap. If hr_y_max < lr_y_min -> no y overlap or hr_y_min > lr_y_max
            # (hr_matrix[:, 0, :] select random row (here 0) since y_max values don't 
            if ((hr_matrix[6, joker, j] <= lr[0, i]) or # hr y_max <= lr y_min
                (hr_matrix[5, joker, j] >= lr[1, i])): # hr y_min >= lr y_max
                # NO y overlap, move to next column.
                j += 1
                # Note: computionally cheaper if we check one first
            elif ((hr_matrix[5, joker, j] < lr[1, i]) & # lr y_min <= hr y_min < lr_y_max
                  (hr_matrix[5, joker, j] >= lr[0, i])):
                # YES y overlap, may be partial or full
                if (hr_matrix[6, joker, j] <= lr[1, i]): # if hr y_max <= lr y_max
                    # YES full y overlap

                    ### x block ###
                    if ((hr_matrix[8, joker, j] <= lr[2, i]) or # hr x_max <= lr x_min
                        (hr_matrix[7, joker, j] >= lr[3, i])): # hr x_min >= lr x_max
                        # NO x overlap, move to next column
                        j += 1
                    elif ((hr_matrix[8, joker, j] > lr[2, i]) &  # hr x_max > lr x_min
                          (hr_matrix[8, joker, j] <= lr[3, i])): # hr x_max <= lr x_max
                        # YES x overlap, may be partial or full
                        if (hr_matrix[7, joker, j] >= lr[2, i]): # hr x_min >= lr x_min
                            # FULL y and FULL x overlap:
                            A = (hr_matrix[6, joker, j] - hr_matrix[5, joker, j]) * (hr_matrix[8, joker, j] - hr_matrix[7, joker, j]) # (hr y_max - hr y_min) * (hr x_max - hr x_min)
                            V = hr_matrix[6, :, j] # Vector of values

                            # Append covariance vector to covariance dictionary
                            covariance_dict[j, i].append(V)
                            # Append Area to weights dict
                            weights_dict[j, i].append(A)

                            j += 1
                        else:
                            # FULL y and partial x overlap (hr extending leftwards of lr in x direction)
                            A = (hr_matrix[6, joker, j] - hr_matrix[5, joker, j]) * (hr_matrix[8, joker, j] - lr[2, i]) # (hr y_max - hr y_min) * (hr x_max - lr x_min)
                            V = hr_matrix[0, :, j] # Vector of values

                            # Append covariance vector to covariance dictionary
                            covariance_dict[j, i].append(V)
                            # Append Area to weights dict
                            weights_dict[j, i].append(A)
                        
                            j += 1
                    elif ((hr_matrix[7, joker, j] >= lr[2, i]) & # lr x_min <= hr x_min < lr x_max
                          (hr_matrix[7, joker, j] < lr[3, i])):
                        # FULL y and partial x overlap: (hr extending rightwards of lr in x direction)

                        A = (hr_matrix[6, joker, j] - hr_matrix[5, joker, j]) * (lr[3, i] - hr_matrix[7, joker, j]) # (hr y_max - hr y_min) * (lr x_max - hr x_min)
                        V = hr_matrix[0, :, j] # Vector of values. channel 0, all rows, current column

                        # Append covariance vector to covariance dictionary
                        covariance_dict[j, i].append(V)
                        # Append Area to weights dict
                        weights_dict[j, i].append(A)

                        j += 1
                    else:
                        print("We didn't catch this case.")
                    ### x block end ###

                else:
                    # YES partial y overlap.
                    
                    ### x block ###
                    if ((hr_matrix[8, joker, j] <= lr[2, i]) or # hr x_max <= lr x_min
                        (hr_matrix[7, joker, j] >= lr[3, i])): # hr x_min >= lr x_max
                        # NO x overlap, move to next column
                        j += 1
                    elif ((hr_matrix[8, joker, j] > lr[2, i]) &  # hr x_max > lr x_min
                          (hr_matrix[8, joker, j] <= lr[3, i])): # hr x_max <= lr x_max
                        # YES x overlap, may be partial or full
                        if (hr_matrix[7, joker, j] >= lr[2, i]): # hr x_min >= lr x_min
                            # Partial y and FULL x overlap (with hr extending upwards of lr)

                            A = (lr[1, i] - hr_matrix[5, joker, j]) * (hr_matrix[8, joker, j] - hr_matrix[7, joker, j]) # (lr y_max - hr y_min) * (hr x_max - hr x_min)
                            V = hr_matrix[0, :, j] # Vector of values. channel 0, all rows, current column

                            # Append covariance vector to covariance dictionary
                            covariance_dict[j, i].append(V)
                            # Append Area to weights dict
                            weights_dict[j, i].append(A)

                            j += 1
                        else:
                            # Partial y and partial x overlap: (with hr extending upwards of lr in y direction, and hr extending leftwards of lr in x direction)

                            A = (lr[1, i] - hr_matrix[5, joker, j]) * (hr_matrix[8, joker, j] - lr[2, i]) # (lr y_max - hr y_min) * (hr x_max - lr x_min)
                            V = hr_matrix[0, :, j] # Vector of values. channel 0, all rows, current column

                            # Append covariance vector to covariance dictionary
                            covariance_dict[j, i].append(V)
                            # Append Area to weights dict
                            weights_dict[j, i].append(A)

                            j += 1
                    elif ((hr_matrix[7, joker, j] >= lr[2, i]) & # lr x_min <= hr x_min < lr x_max
                          (hr_matrix[7, joker, j] < lr[3, i])):
                        # Partial y and partial x overlap: (with hr extenting upwards in y direction, with hr extending rightwards of lr in x sirection)

                        A = (lr[1, i] - hr_matrix[5, joker, j]) * (lr[3, i] - hr_matrix[7, joker, j]) # (lr y_max - hr y_min) * (lr x_max - hr x_min)
                        V = hr_matrix[0, :, j] # Vector of values. channel 0, all rows, current column

                        # Append covariance vector to covariance dictionary
                        covariance_dict[j, i].append(V)
                        # Append Area to weights dict
                        weights_dict[j, i].append(A)
                    
                        j += 1
                    else:
                        print("We didn't catch this case.")
                    ### x block end ###

            elif ((hr_matrix[6, joker, j] > lr[0, i]) & # lr y_min < hr y_max <= lr_y_max
                  (hr_matrix[6, joker, j] <= lr[1, i])):
                  # YES partial y overlap.
                
                    ### x block ###
                    if ((hr_matrix[8, joker, j] <= lr[2, i]) or # hr x_max <= lr x_min
                        (hr_matrix[7, joker, j] >= lr[3, i])): # hr x_min >= lr x_max
                        # NO x overlap, move to next column
                        j += 1
                    elif ((hr_matrix[8, joker, j] > lr[2, i]) &  # hr x_max > lr x_min
                          (hr_matrix[8, joker, j] <= lr[3, i])): # hr x_max <= lr x_max
                        # YES x overlap, may be partial or full
                        if (hr_matrix[7, joker, j] >= lr[2, i]): # hr x_min >= lr x_min
                            # Partial y and FULL x overlap: (with hr extending downwards of lr in y direction)
                            
                            A = (hr_matrix[6, joker, j] - lr[0, i]) * (hr_matrix[8, joker, j] - hr_matrix[7, joker, j]) # (hr y_max - lr y_min) * (hr x_max - hr x_min)
                            V = hr_matrix[0, :, j] # Vector of values. channel 0, all rows, current column

                            # Append covariance vector to covariance dictionary
                            covariance_dict[j, i].append(V)
                            # Append Area to weights dict
                            weights_dict[j, i].append(A)

                            j += 1

                        else:
                            # Partial y and partial x overlap: (with hr extending downwards of lr in y direction, and with hr extending leftwars in x direction)
                            A = (hr_matrix[6, joker, j] - lr[0, i]) * (hr_matrix[8, joker, j] - lr[2, i]) # (hr y_max - lr y_min) * (hr x_max - lr x_min)
                            V = hr_matrix[0, :, j] # Vector of values. channel 0, all rows, current column

                            # Append covariance vector to covariance dictionary
                            covariance_dict[j, i].append(V)
                            # Append Area to weights dict
                            weights_dict[j, i].append(A)
                    
                            j += 1
                    elif ((hr_matrix[7, joker, j] >= lr[2, i]) & # lr x_min <= hr x_min < lr x_max
                          (hr_matrix[7, joker, j] < lr[3, i])):
                        # Partial y and partial x overlap: (with hr extending downwards of lr in y direction, with hr extending rightwards of lr in x direction)

                        A = (hr_matrix[6, joker, j] - lr[0, i]) * (lr[3, i] - hr_matrix[7, joker, j]) # (hr y_max - lr y_min) * (lr x_max - hr x_min)
                        V = hr_matrix[0, :, j] # Vector of values. channel 0, all rows, current column

                        # Append covariance vector to covariance dictionary
                        covariance_dict[j, i].append(V)
                        # Append Area to weights dict
                        weights_dict[j, i].append(A)
                    
                        j += 1
                    else:
                        print("We didn't catch this case.")
                    ### x block end ###

            else:
                print("We didn't catch this case.")
        
        return covariance_dict, weights_dict


In [80]:
lr_box_tensor.shape

torch.Size([4, 25, 25])

In [19]:
covariance_dict, weights_dict = compute_k_ah_al_rectilinear(base_covar_box, lr_box_tensor)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [66]:
hr_dims_flat = np.array(base_covar_box.shape)[-1]
lr_dims_flat = np.array(lr_box_tensor.reshape(4, -1).shape)[-1]

print(lr_dims_flat)
print(hr_dims_flat)

625
2116


In [73]:
covariance_dict = {(hr, lr): [] for hr in range(0, hr_dims_flat) for lr in range(0, lr_dims_flat)}

In [74]:
covariance_dict[0,1].append(torch.ones(size = (2,1)))

In [76]:
covariance_dict[0,1].append(torch.zeros(size = (2,1)))

In [77]:
covariance_dict[0,1]

[tensor([[1.],
         [1.]]),
 tensor([[0.],
         [0.]])]